# Train Your Own LLM From Scratch
**No pretrained models. No HuggingFace tokenizers. Just PyTorch.**

This notebook:
1. Clones your GitHub repo
2. Installs PyTorch (only dependency)
3. Uploads your corpus
4. Trains a BPE tokenizer from scratch
5. Trains a GPT model from scratch
6. Generates text with the trained model
7. Downloads your checkpoint

> **Runtime → Change runtime type → T4 GPU** before running.

## Step 0 — Verify GPU

In [ ]:
import torch
print('PyTorch version:', torch.__version__)
print('CUDA available :', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU            :', torch.cuda.get_device_name(0))
    print('VRAM           :', round(torch.cuda.get_device_properties(0).total_memory/1e9, 1), 'GB')

## Step 1 — Clone your GitHub repo

In [ ]:
# TODO: replace with your actual GitHub repo URL
GITHUB_REPO = 'https://github.com/YOUR_USERNAME/YOUR_REPO.git'

!git clone {GITHUB_REPO} llm
%cd llm
!ls -la

## Step 2 — Install the only dependency

In [ ]:
# Colab already has PyTorch; this just confirms / upgrades if needed
!pip install -q -r requirements.txt
import torch; print('torch', torch.__version__)

## Step 3 — Upload your corpus
Upload any plain UTF-8 `.txt` file (books, Wikipedia dumps, code, etc.).
Bigger corpus → better model. Minimum ~1 MB recommended.

In [ ]:
from google.colab import files
uploaded = files.upload()   # a file picker will appear

import os
CORPUS_PATH = list(uploaded.keys())[0]
size_mb = os.path.getsize(CORPUS_PATH) / 1e6
print(f'Corpus: {CORPUS_PATH}  ({size_mb:.1f} MB)')

## Step 4 — Configure training
Adjust these to match your corpus size and available VRAM.

| Corpus size | Recommended vocab_size | batch_size | max_steps |
|-------------|------------------------|------------|-----------|
| < 1 MB      | 1000–2000              | 16         | 2000      |
| 1–10 MB     | 2000–4000              | 32         | 5000      |
| 10–100 MB   | 4000–8000              | 64         | 20000     |
| > 100 MB    | 8000–16000             | 128        | 50000+    |

In [ ]:
# ── Edit these ────────────────────────────────────────────────────────
VOCAB_SIZE   = 4096   # BPE vocabulary size
CONTEXT_LEN  = 256    # tokens per sample (GPT context window)
D_MODEL      = 512    # embedding dimension
N_HEADS      = 8      # attention heads
N_LAYERS     = 6      # transformer blocks
D_FF         = 2048   # feed-forward hidden size
BATCH_SIZE   = 32     # samples per batch
MAX_STEPS    = 5000   # total gradient steps
MAX_LR       = 3e-4   # peak learning rate
WARMUP_STEPS = 200    # linear LR warm-up
SAVE_DIR     = 'checkpoints'
# ─────────────────────────────────────────────────────────────────────

print('Config set. Approximate parameter count:')
approx = (VOCAB_SIZE * D_MODEL) + N_LAYERS * (4 * D_MODEL**2 + 2 * D_MODEL * D_FF)
print(f'  ~{approx/1e6:.1f} M parameters')

## Step 5 — Train the tokenizer + model

In [ ]:
!python train.py \
    --corpus        {CORPUS_PATH} \
    --save_dir      {SAVE_DIR} \
    --vocab_size    {VOCAB_SIZE} \
    --context_len   {CONTEXT_LEN} \
    --d_model       {D_MODEL} \
    --n_heads       {N_HEADS} \
    --n_layers      {N_LAYERS} \
    --d_ff          {D_FF} \
    --batch_size    {BATCH_SIZE} \
    --max_steps     {MAX_STEPS} \
    --max_lr        {MAX_LR} \
    --warmup_steps  {WARMUP_STEPS} \
    --log_every     50 \
    --val_every     500

## Step 6 — Plot the training loss

In [ ]:
import csv, matplotlib.pyplot as plt

train_steps, train_losses = [], []
val_steps,   val_losses   = [], []

with open(f'{SAVE_DIR}/training_log.csv') as f:
    for row in csv.DictReader(f):
        if row['train_loss']:
            train_steps.append(int(row['step']))
            train_losses.append(float(row['train_loss']))
        if row['val_loss']:
            val_steps.append(int(row['step']))
            val_losses.append(float(row['val_loss']))

plt.figure(figsize=(10, 4))
plt.plot(train_steps, train_losses, alpha=0.6, label='train loss')
plt.plot(val_steps,   val_losses,   marker='o', label='val loss')
plt.xlabel('Step'); plt.ylabel('Cross-entropy loss')
plt.title('Training curve'); plt.legend(); plt.grid(True)
plt.tight_layout(); plt.show()

import math
if val_losses:
    print(f'Best val loss : {min(val_losses):.4f}  (perplexity {math.exp(min(val_losses)):.2f})')

## Step 7 — Generate text

In [ ]:
import sys, torch
sys.path.insert(0, '.')

from tokenizer import BPETokenizer
from model     import GPT, GPTConfig
import json
from pathlib import Path

device    = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
tokenizer = BPETokenizer.load(f'{SAVE_DIR}/tokenizer.json')
cfg_dict  = json.loads(Path(f'{SAVE_DIR}/model_config.json').read_text())
cfg       = GPTConfig(**{k: v for k, v in cfg_dict.items() if hasattr(GPTConfig, k)})

model = GPT(cfg)
model.load_state_dict(torch.load(f'{SAVE_DIR}/best_model.pt', map_location='cpu'))
model.to(device).eval()

print(f'Model loaded: {model.num_parameters()/1e6:.1f}M params on {device}')

In [ ]:
def generate(prompt, max_new_tokens=150, temperature=0.9, top_k=50, top_p=0.95):
    ids = tokenizer.encode(prompt, add_bos=True)
    inp = torch.tensor([ids], dtype=torch.long, device=device)
    with torch.inference_mode():
        out = model.generate(
            inp,
            max_new_tokens=max_new_tokens,
            temperature=temperature,
            top_k=top_k,
            top_p=top_p,
            eos_id=tokenizer.eos_id,
        )
    new_ids = out[0, len(ids):].tolist()
    return prompt + tokenizer.decode(new_ids)

# ── Edit your prompt here ─────────────────────────────────────────────
PROMPT = 'Once upon a time'

print(generate(PROMPT))

## Step 8 — Download your checkpoint

In [ ]:
import shutil
from google.colab import files

# Zip up the checkpoints folder and download it
shutil.make_archive('my_llm_checkpoint', 'zip', SAVE_DIR)
files.download('my_llm_checkpoint.zip')